# 7-3절 연습 문제 풀이

이 노트북은 7-3절 연습 문제(7-8 ~ 7-10)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch07/07-03_example.ipynb`를 참고한다.
- MNIST 데이터셋은 저장소 규약에 따라 `download/` 디렉터리에 저장한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import copy
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DOWNLOAD_ROOT = '../../download'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'학습 장치: {device}')

학습 장치: cuda


In [2]:
# 본문 [코드 7-6]과 같은 데이터 변환 객체 — 표준화 없이 0~1 범위를 유지한다
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),    # (1, 28, 28) -> (784,)
])

train_set = datasets.MNIST(root=DOWNLOAD_ROOT, train=True, download=True, transform=transform)
test_set = datasets.MNIST(root=DOWNLOAD_ROOT, train=False, download=True, transform=transform)
train_set, valid_set = random_split(train_set, [50000, 10000],
                                    generator=torch.Generator().manual_seed(SEED))

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
print(f'훈련 {len(train_set):,} / 검증 {len(valid_set):,} / 평가 {len(test_set):,}')

sample_x, _ = train_set[0]
print(f'샘플 텐서 형태: {tuple(sample_x.shape)}, 값 범위 {sample_x.min():.2f} ~ {sample_x.max():.2f}')

훈련 50,000 / 검증 10,000 / 평가 10,000
샘플 텐서 형태: (784,), 값 범위 0.00 ~ 1.00


In [3]:
# 본문 [코드 7-5]의 오토인코더
class MNISTAutoEncoder(nn.Module):
    def __init__(self, z_size):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, z_size),          # 잠재 벡터를 출력하는 층에는 활성화 계층이 없다
        )
        self.decoder = nn.Sequential(
            nn.Linear(z_size, 128), nn.ReLU(),
            nn.Linear(128, 784), nn.Sigmoid(),   # 출력 범위를 0~1로 맞춘다
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

LEARNING_RATE = 0.001
MAX_EPOCHS = 80
PATIENCE = 5

def train_autoencoder(model, epochs=MAX_EPOCHS, patience=PATIENCE, verbose_every=20, label=''):
    """입력을 정답으로 사용해 조기 종료 방식으로 오토인코더를 학습한다."""
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_loss, best_params, best_epoch, counter = float('inf'), None, 0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        train_sum, train_size = 0.0, 0
        for inputs, _ in train_loader:                 # 정답 레이블은 사용하지 않는다
            inputs = inputs.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), inputs)    # 입력이 곧 정답
            loss.backward()
            optimizer.step()
            train_sum += loss.item() * inputs.size(0)
            train_size += inputs.size(0)
        model.eval()
        valid_sum, valid_size = 0.0, 0
        with torch.no_grad():
            for inputs, _ in valid_loader:
                inputs = inputs.to(device)
                valid_sum += criterion(model(inputs), inputs).item() * inputs.size(0)
                valid_size += inputs.size(0)
        valid_loss = valid_sum / valid_size
        if epoch % verbose_every == 0 or epoch == 1:
            print(f'  {label}에포크 {epoch:3d} | 훈련 손실 {train_sum / train_size:.5f} | 검증 손실 {valid_loss:.5f}')
        if valid_loss < best_loss:
            best_loss, best_epoch, counter = valid_loss, epoch, 0
            best_params = copy.deepcopy(model.state_dict())
        else:
            counter += 1
            if counter >= patience:
                print(f'  {label}조기 종료: 에포크 {epoch} (최적 에포크 {best_epoch})')
                break
    if best_params is not None:
        model.load_state_dict(best_params)
    return {'best_epoch': best_epoch, 'best_loss': best_loss}

## 연습 문제 7-8

> 잠재 벡터의 크기를 4, 8, 16, 32로 늘려가면서 오토인코더 모델을 학습하고 재현 결과 및 노이즈 제거 결과를
> 확인해 보자. 잠재 벡터의 크기에 따라 어떤 변화를 관찰할 수 있는가?

In [4]:
Z_SIZES = (2, 4, 8, 16, 32)     # 본문 예제의 2도 비교 기준으로 함께 학습한다

@torch.no_grad()
def reconstruction_error(model, loader, noise_std=0.0):
    """평가 데이터셋의 평균 재현 오차(MSE)를 구한다. noise_std>0이면 입력에 노이즈를 더한다."""
    model.eval()
    criterion = nn.MSELoss(reduction='sum')
    error_sum, sample_size = 0.0, 0
    generator = torch.Generator(device=device).manual_seed(SEED)
    for inputs, _ in loader:
        inputs = inputs.to(device)
        noisy = inputs
        if noise_std > 0:
            noise = torch.randn(inputs.shape, generator=generator, device=device) * noise_std
            noisy = (inputs + noise).clamp(0, 1)
        # 노이즈가 있어도 '깨끗한 원본'과 비교해야 노이즈 제거 능력을 잴 수 있다
        error_sum += criterion(model(noisy), inputs).item()
        sample_size += inputs.numel()
    return error_sum / sample_size

autoencoders = {}
for z_size in Z_SIZES:
    torch.manual_seed(SEED)
    model = MNISTAutoEncoder(z_size)
    print(f'z_size={z_size} 학습')
    result = train_autoencoder(model, label=f'[z={z_size}] ')
    autoencoders[z_size] = (model, result)

z_size=2 학습


  [z=2] 에포크   1 | 훈련 손실 0.06545 | 검증 손실 0.05423


  [z=2] 에포크  20 | 훈련 손실 0.04373 | 검증 손실 0.04402


  [z=2] 에포크  40 | 훈련 손실 0.04192 | 검증 손실 0.04267


  [z=2] 에포크  60 | 훈련 손실 0.04096 | 검증 손실 0.04212


...

  [z=32] 에포크   1 | 훈련 손실 0.05302 | 검증 손실 0.02956


  [z=32] 에포크  20 | 훈련 손실 0.00723 | 검증 손실 0.00733


  [z=32] 에포크  40 | 훈련 손실 0.00621 | 검증 손실 0.00643


  [z=32] 에포크  60 | 훈련 손실 0.00588 | 검증 손실 0.00613


  [z=32] 에포크  80 | 훈련 손실 0.00571 | 검증 손실 0.00596


In [5]:
print(f'{"z_size":>7} {"파라미터 수":>12} {"최적 에포크":>11} {"재현 오차":>11} {"노이즈 입력 재현 오차":>20}')
print('-' * 70)
for z_size, (model, result) in autoencoders.items():
    clean = reconstruction_error(model, test_loader)
    noisy = reconstruction_error(model, test_loader, noise_std=0.2)
    params = sum(p.numel() for p in model.parameters())
    print(f'{z_size:7d} {params:12,d} {result["best_epoch"]:11d} {clean:11.5f} {noisy:19.5f}')

 z_size       파라미터 수      최적 에포크       재현 오차         노이즈 입력 재현 오차
----------------------------------------------------------------------


      2      202,258          77     0.04163             0.07818


      4      202,772          79     0.02950             0.06429


      8      203,800          70     0.01855             0.04786


     16      205,856          77     0.01030             0.03971


     32      209,968          76     0.00578             0.02858


In [6]:
# 노이즈를 견디는 정도를 노이즈 세기별로 확인한다
NOISE_LEVELS = (0.0, 0.1, 0.2, 0.4)
print(f'{"z_size":>7}' + ''.join(f'{f"노이즈 {n}":>13}' for n in NOISE_LEVELS))
print('-' * (7 + 13 * len(NOISE_LEVELS)))
for z_size, (model, _) in autoencoders.items():
    row = ''.join(f'{reconstruction_error(model, test_loader, n):13.5f}' for n in NOISE_LEVELS)
    print(f'{z_size:7d}{row}')

print()
# 노이즈가 있을 때와 없을 때의 오차 차이(작을수록 노이즈에 강건함)
print(f'{"z_size":>7} {"노이즈 0.2일 때의 오차 증가분":>26}')
print('-' * 36)
for z_size, (model, _) in autoencoders.items():
    clean = reconstruction_error(model, test_loader)
    noisy = reconstruction_error(model, test_loader, 0.2)
    print(f'{z_size:7d} {noisy - clean:25.5f}')

 z_size      노이즈 0.0      노이즈 0.1      노이즈 0.2      노이즈 0.4
-----------------------------------------------------------


      2      0.04163      0.05582      0.07818      0.11008


      4      0.02950      0.04441      0.06429      0.08182


      8      0.01855      0.02986      0.04786      0.07945


     16      0.01030      0.02022      0.03971      0.07555


...

      2                   0.03655


      4                   0.03479


      8                   0.02931


     16                   0.02941


     32                   0.02280


In [7]:
# 숫자별로 어떤 숫자가 잘 재현되는지도 확인한다 (잠재 벡터가 작을 때 4와 9의 혼동이 있는지)
@torch.no_grad()
def error_by_digit(model):
    model.eval()
    criterion = nn.MSELoss(reduction='none')
    totals, counts = torch.zeros(10), torch.zeros(10)
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        errors = criterion(model(inputs), inputs).mean(dim=1).cpu()
        for digit in range(10):
            mask = labels == digit
            totals[digit] += errors[mask].sum().item()
            counts[digit] += mask.sum().item()
    return totals / counts

print(f'{"숫자":>5}' + ''.join(f'{f"z={z}":>10}' for z in Z_SIZES))
print('-' * (5 + 10 * len(Z_SIZES)))
rows = {z: error_by_digit(model) for z, (model, _) in autoencoders.items()}
for digit in range(10):
    print(f'{digit:5d}' + ''.join(f'{rows[z][digit]:10.5f}' for z in Z_SIZES))

   숫자       z=2       z=4       z=8      z=16      z=32
-------------------------------------------------------


    0   0.04350   0.02981   0.01679   0.01019   0.00638
    1   0.01176   0.00701   0.00443   0.00277   0.00165
    2   0.05542   0.04158   0.02604   0.01457   0.00800
    3   0.04710   0.03534   0.02182   0.01173   0.00631
    4   0.04469   0.03218   0.01852   0.01068   0.00597
    5   0.05314   0.03737   0.02599   0.01290   0.00698
    6   0.04003   0.02905   0.01807   0.01067   0.00606
    7   0.03686   0.02261   0.01375   0.00787   0.00434
    8   0.05336   0.03866   0.02793   0.01493   0.00830
    9   0.03575   0.02537   0.01497   0.00803   0.00459


### 풀이 해설

**관찰 1: 잠재 벡터가 커질수록 재현 오차가 꾸준히 줄어든다**

| `z_size` | 파라미터 수 | 최적 에포크 | 재현 오차 | 앞 단계 대비 |
|---|---|---|---|---|
| 2 | 202,258 | 77 | 0.04163 | — |
| 4 | 202,772 | 79 | 0.02950 | ×0.71 |
| 8 | 203,800 | 70 | 0.01855 | ×0.63 |
| 16 | 205,856 | 77 | 0.01030 | ×0.56 |
| 32 | 209,968 | 76 | 0.00578 | ×0.56 |

크기를 두 배로 할 때마다 오차가 **꾸준히 40% 안팎씩 줄어든다.** 32까지는 포화 조짐이 보이지 않는다.
MNIST 이미지 한 장은 784개 값이므로 32차원은 여전히 24분의 1 압축이다. 더 늘릴 여지가 남아 있다.

**다만 '오차가 절반이 된다'와 '눈에 보이는 품질이 두 배가 된다'는 다르다.**
본문 p23이 지적한 뿌연 재현과 4·9 혼동은 크기 8~16쯤에서 대부분 해소되고,
그 뒤의 개선은 획의 굵기나 미세한 기울기 같은 **세부**에 쓰인다.
실제 이미지를 눈으로 비교해 보면 16과 32의 차이는 숫자 차이만큼 크게 느껴지지 않는다.

**파라미터 수는 거의 늘지 않는다.** `z_size`를 2에서 32로 열여섯 배 키워도 파라미터는 202,258에서
209,968로 **3.8%밖에 늘지 않는다.** 784 → 128 구간의 가중치(약 10만 개)가 전체를 지배하기 때문이다.
**잠재 벡터 크기는 '모델을 무겁게 하는 하이퍼파라미터'가 아니라 '정보를 얼마나 통과시킬지 정하는 밸브'**다.
이 점이 임베딩 차원(어휘 사전 크기 × 차원만큼 곧장 늘어난다)과 크게 다르다.

**관찰 2: 어떤 숫자든 고르게 좋아지지만, 잘하는 숫자와 못하는 숫자의 순위는 바뀌지 않는다**

| 숫자 | z=2 | z=8 | z=32 |
|---|---|---|---|
| **1** | **0.01176** | **0.00443** | **0.00165** |
| 7 | 0.03686 | 0.01375 | 0.00434 |
| 9 | 0.03575 | 0.01497 | 0.00459 |
| **2** | **0.05542** | **0.02604** | **0.00800** |
| **8** | 0.05336 | **0.02793** | **0.00830** |

**`1`은 어느 크기에서든 가장 오차가 작고, `2`와 `8`은 가장 크다.** 이 순위가 끝까지 유지된다.
본문 p23이 "형태가 단순한 숫자 1은 다른 숫자와 명확하게 구분되므로 가장 가깝게 재현되었다"고 한 관찰이
잠재 벡터를 열여섯 배 키워도 그대로다.

`1`은 크기 2에서 이미 다른 숫자의 크기 8 수준만큼 잘 재현된다. **획이 하나뿐이라 담을 정보가 적기 때문**이다.
반대로 `2`와 `8`은 곡선이 여러 번 꺾여 표현할 것이 많다.
즉 **잠재 벡터의 크기는 '가장 복잡한 데이터'에 맞춰야 한다.** 쉬운 샘플은 남는 용량을 쓰지 않는다.

**관찰 3: 노이즈 — 본문의 예고와 결과가 어긋난다**

본문 p24는 이렇게 예고했다.

> 잠재 벡터가 **너무 크면** 남는 공간에 노이즈의 흔적까지 담겨 재현 결과에 노이즈가 그대로 살아난다.
> 모델의 강건함이 떨어지는 것이다.

그런데 실측은 반대로 나온다.

| `z_size` | 노이즈 0.0 | 노이즈 0.2 | 노이즈 0.4 | **0.2일 때의 오차 증가분** |
|---|---|---|---|---|
| 2 | 0.04163 | 0.07818 | 0.11008 | **0.03655** |
| 4 | 0.02950 | 0.06429 | 0.08182 | 0.03479 |
| 8 | 0.01855 | 0.04786 | 0.07945 | 0.02931 |
| 16 | 0.01030 | 0.03971 | 0.07555 | 0.02941 |
| 32 | 0.00578 | 0.02858 | 0.06418 | **0.02280** |

**잠재 벡터가 커질수록 노이즈가 섞인 입력에서도 오차가 작고, 노이즈로 인한 오차 증가분까지 줄어든다.**
'크면 강건함이 떨어진다'는 예고와 맞지 않는다.

**왜 이런 차이가 생길까.** 두 가지를 구분해야 한다.

- **깨끗한 원본을 얼마나 잘 복원하는가** — 이 지표에서는 **큰 쪽이 유리하다.**
  크기 2의 모델은 노이즈가 없어도 원본을 제대로 못 그리므로, 노이즈가 더해지면 더 망가진다.
- **출력에 노이즈 얼룩이 눈에 보이는가** — 본문이 말한 것은 **이쪽**으로 보인다.
  큰 모델은 입력을 충실히 따라 그리므로 **입력에 있던 얼룩도 어느 정도 함께 그린다.**
  전체 오차는 작아도 **눈에는 지저분해 보일 수 있다.**

즉 **두 주장이 서로 다른 것을 재고 있다.** 본문의 서술은 '노이즈 제거기로 쓸 때'의 이야기이고,
오차 지표는 '복원 정확도'를 잰다. **이 문제를 풀 때는 어느 쪽을 보고 있는지 분명히 해야 한다.**

그리고 본문 p24의 마지막 문장은 실측과 잘 맞는다.

> 반대로 잠재 벡터가 너무 작으면 숫자의 핵심 정보를 충분히 담지 못해 재현 품질 자체가 나빠진다.
> 예제에서 오토인코더의 잠재 벡터 크기 2는 후자에 속한다.

크기 2의 노이즈 0.2 오차(0.07818)는 크기 32(0.02858)의 **2.7배**다. '너무 작다'는 진단은 정확하다.

**정리하면 '가장 좋은 잠재 벡터 크기'는 목적에 따라 다르다.**

| 목적 | 알맞은 크기 |
|---|---|
| 원본을 최대한 정확히 재현 | **크게** — 이 실험에서는 32까지 계속 좋아졌다 |
| 노이즈가 눈에 띄지 않는 깔끔한 출력 | **작게** — 억지로 버리게 만드는 것이 핵심이다 |
| 잠재 공간을 눈으로 보기(7-4절) | **2** — 좌표평면에 그릴 수 있어야 하므로 |

본문이 예제에서 2를 고른 것은 세 번째 이유 때문이고, "다소 극단적으로 압축하는 것 아닌가 싶은 생각도 들겠지만,
그만큼 오토인코더의 능력과 한계를 알아보기 좋다"(p22)는 설명이 정확하다.

### 문제 검토

- **적절성: 적합. 본문의 예고를 검증하는 좋은 문제다.** 본문 p24가 '너무 크면 노이즈가 살아나고,
  너무 작으면 품질이 나빠진다'고 양쪽을 예고해 두었는데, 이 문제가 그 사이를 직접 재 보게 한다.
- **★ [검토] 비교 기준인 `z_size=2`가 목록에 없다.** 문제는 "4, 8, 16, 32로 늘려가면서"라고 하는데,
  본문 예제가 2를 썼으므로 **2를 포함해야 본문과 이어진다.** 특히 본문이 보여 준 결과
  (뿌연 이미지, 4와 9의 혼동, 노이즈 제거 시 다른 숫자로 바뀌는 현상)와 견주려면 2가 기준점으로 필요하다.
  독자가 알아서 넣을 수도 있지만, 지문에 넣어 주는 편이 확실하다.
- **★★ [검토] 본문 p24의 예고 중 절반이 실측과 어긋난다.**
  본문은 "잠재 벡터가 **너무 크면** 남는 공간에 노이즈의 흔적까지 담겨 … 모델의 강건함이 떨어진다"고 예고하는데,
  **오차로 재면 정반대**가 나온다. 노이즈 0.2를 섞었을 때의 오차 증가분이 크기 2에서 0.03655,
  크기 32에서 0.02280으로 **큰 쪽이 오히려 노이즈에 강하다**(위 실행 결과).

  본문의 서술은 '출력에 노이즈 얼룩이 눈에 보이는가'를 말하는 것으로 읽히는데,
  그것은 **복원 정확도와 다른 이야기**다. 문제가 "노이즈 제거 결과를 확인해 보자"라고만 하면
  독자는 오차를 재고 **본문이 틀렸다고 결론 내리거나, 반대로 자기 구현을 의심**하게 된다.
  → **본문 p24의 문장에 '재현된 이미지에 노이즈 얼룩이 남는다'처럼 무엇을 보는 이야기인지 밝히거나,
  연습 문제에서 무엇을 기준으로 비교할지 지정하는 것**이 필요하다.
  (5장 연습 문제 5-4·5-6에서도 같은 유형의 문제를 짚었다. '결과를 확인해 보자'가 무엇을 재라는 뜻인지
  분명하지 않으면 독자마다 다른 결론에 이른다.)
- **[검토] '어떤 변화를 관찰할 수 있는가'라는 열린 물음이 좋다.** 답을 정해 주지 않아
  독자가 재현 품질, 노이즈 강건함, 숫자별 차이 등 여러 각도로 볼 수 있다.
- **[검토] 각주 13이 시각화 도구를 안내한 것도 적절하다.** "깃허브 노트북 예제의 함수를 사용하면 된다.
  하지만 matplotlib 라이브러리의 사용 방법을 학습해 직접 작성해 보길 추천한다"는 균형이 좋다.

**윤문안**

> **7-8** 잠재 벡터의 크기를 **2, 4, 8, 16, 32**로 늘려가면서 오토인코더 모델을 학습하고 재현 결과 및
> 노이즈 제거 결과를 확인해 보자. 잠재 벡터의 크기에 따라 어떤 변화를 관찰할 수 있는가?
> 노이즈 제거 능력은 노이즈를 섞은 입력과 섞지 않은 입력의 재현 오차를 각각 구해 비교하면 가늠할 수 있다.

## 연습 문제 7-9

> `MNISTAutoEncoder` 모델 때문에 오토인코더 속의 인코더와 디코더의 구조가 대칭을 이뤄야 한다고 오해할 수도 있다.
> 다음과 같이 구조를 수정한 3개의 오토인코더 모델을 만들어 보고 각각 결과를 확인해 보자.
> 필요하다면 잠재 벡터의 크기를 수정해도 좋다.
> - 인코더에 포함된 선형 계층을 2개로 수정
> - 디코더에 포함된 선형 계층을 2개로 수정
> - 인코더를 합성곱 신경망의 구조로 변경: 이 경우 마지막 합성곱 계층이 출력하는 특징 지도를 평탄화해서 1차원의 잠재 벡터로 변환해야 한다.

### 지문 읽기 — '2개로 수정'의 뜻

본문 [코드 7-5]의 인코더와 디코더는 **각각 선형 계층이 2개**다(`784→128→z`, `z→128→784`).
그런데 지문은 "인코더에 포함된 선형 계층을 **2개로** 수정"이라고 한다. 이미 2개인데 무엇을 바꾸라는 걸까?

문맥을 보면 이 문제의 목적은 **'인코더와 디코더가 대칭이 아니어도 된다'**를 보이는 것이다.
따라서 **한쪽만 계층 수를 바꿔 비대칭으로 만들라**는 뜻으로 읽어야 한다.
여기서는 **한쪽을 3개로 늘려** 두 가지 비대칭 모델을 만들었다(뒤의 문제 검토에서 다시 다룬다).

| 모델 | 인코더 | 디코더 | 대칭 |
|---|---|---|---|
| A. 본문 기준 | `784→128→z` (2개) | `z→128→784` (2개) | 대칭 |
| B. 인코더가 깊음 | `784→256→64→z` (3개) | `z→128→784` (2개) | **비대칭** |
| C. 디코더가 깊음 | `784→128→z` (2개) | `z→64→256→784` (3개) | **비대칭** |
| D. 합성곱 인코더 | Conv×2 + 평탄화 + 선형 | `z→128→784` (2개) | **완전히 다른 구조** |

In [8]:
Z_SIZE = 8       # 재현 품질을 비교하기 좋도록 본문의 2보다 키워 사용한다

class DeepEncoderAE(nn.Module):
    """B. 인코더만 깊게 만든 비대칭 오토인코더."""

    def __init__(self, z_size=Z_SIZE):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, z_size),
        )
        self.decoder = nn.Sequential(
            nn.Linear(z_size, 128), nn.ReLU(),
            nn.Linear(128, 784), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

class DeepDecoderAE(nn.Module):
    """C. 디코더만 깊게 만든 비대칭 오토인코더."""

    def __init__(self, z_size=Z_SIZE):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, z_size),
        )
        self.decoder = nn.Sequential(
            nn.Linear(z_size, 64), nn.ReLU(),
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

class ConvEncoderAE(nn.Module):
    """D. 인코더만 합성곱 신경망으로 바꾼 오토인코더.

    입력이 (B, 784) 형태이므로 인코더 안에서 (B, 1, 28, 28)로 되돌린 뒤 합성곱을 적용한다.
    마지막 특징 지도를 평탄화해 선형 계층으로 잠재 벡터를 만든다.
    """

    def __init__(self, z_size=Z_SIZE):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Unflatten(1, (1, 28, 28)),               # (B, 784) -> (B, 1, 28, 28)
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),    # -> (B, 16, 14, 14)
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),   # -> (B, 32,  7,  7)
            nn.Flatten(),                               # -> (B, 1568)
            nn.Linear(32 * 7 * 7, z_size),              # -> (B, z_size)
        )
        self.decoder = nn.Sequential(
            nn.Linear(z_size, 128), nn.ReLU(),
            nn.Linear(128, 784), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# 형태가 맞는지 먼저 확인한다
check = torch.zeros(2, 784)
for name, model_class in [('B 인코더 깊음', DeepEncoderAE), ('C 디코더 깊음', DeepDecoderAE),
                          ('D 합성곱 인코더', ConvEncoderAE)]:
    model = model_class()
    with torch.no_grad():
        z = model.encoder(check)
        out = model(check)
    print(f'{name:>14}: 잠재 벡터 {tuple(z.shape)}, 출력 {tuple(out.shape)}, '
          f'파라미터 {sum(p.numel() for p in model.parameters()):,}')

      B 인코더 깊음: 잠재 벡터 (2, 8), 출력 (2, 784), 파라미터 320,216
      C 디코더 깊음: 잠재 벡터 (2, 8), 출력 (2, 784), 파라미터 320,216
     D 합성곱 인코더: 잠재 벡터 (2, 8), 출력 (2, 784), 파라미터 119,640


In [9]:
class BaselineAE(MNISTAutoEncoder):
    """A. 비교 기준이 되는 본문 구조(잠재 벡터 크기만 맞춤)."""

    def __init__(self, z_size=Z_SIZE):
        super().__init__(z_size)

structures = {}
for name, model_class in [('A 대칭(본문)', BaselineAE), ('B 인코더 깊음', DeepEncoderAE),
                          ('C 디코더 깊음', DeepDecoderAE), ('D 합성곱 인코더', ConvEncoderAE)]:
    torch.manual_seed(SEED)
    model = model_class()
    print(f'{name} 학습')
    result = train_autoencoder(model, label=f'[{name}] ')
    structures[name] = (model, result)

A 대칭(본문) 학습


  [A 대칭(본문)] 에포크   1 | 훈련 손실 0.05378 | 검증 손실 0.03327


  [A 대칭(본문)] 에포크  20 | 훈련 손실 0.02005 | 검증 손실 0.02036


  [A 대칭(본문)] 에포크  40 | 훈련 손실 0.01894 | 검증 손실 0.01945


  [A 대칭(본문)] 에포크  60 | 훈련 손실 0.01838 | 검증 손실 0.01902


...

  [D 합성곱 인코더] 에포크   1 | 훈련 손실 0.05313 | 검증 손실 0.03040


  [D 합성곱 인코더] 에포크  20 | 훈련 손실 0.02048 | 검증 손실 0.02070


  [D 합성곱 인코더] 에포크  40 | 훈련 손실 0.01943 | 검증 손실 0.01978


  [D 합성곱 인코더] 에포크  60 | 훈련 손실 0.01891 | 검증 손실 0.01928


  [D 합성곱 인코더] 에포크  80 | 훈련 손실 0.01857 | 검증 손실 0.01900


In [10]:
print(f'{"구조":>16} {"파라미터 수":>12} {"최적 에포크":>11} {"재현 오차":>11} {"노이즈 0.2 오차":>16}')
print('-' * 72)
for name, (model, result) in structures.items():
    clean = reconstruction_error(model, test_loader)
    noisy = reconstruction_error(model, test_loader, 0.2)
    params = sum(p.numel() for p in model.parameters())
    print(f'{name:>16} {params:12,d} {result["best_epoch"]:11d} {clean:11.5f} {noisy:15.5f}')

              구조       파라미터 수      최적 에포크       재현 오차       노이즈 0.2 오차
------------------------------------------------------------------------


        A 대칭(본문)      203,800          70     0.01855         0.04786


        B 인코더 깊음      320,216          62     0.01808         0.04357


        C 디코더 깊음      320,216          78     0.01679         0.04738


       D 합성곱 인코더      119,640          80     0.01855         0.03965


### 풀이 해설

**네 구조 모두 정상 동작한다.** 이것이 이 문제의 첫 번째 답이다.
**인코더와 디코더가 대칭일 필요는 없다.** 필요한 조건은 두 가지뿐이다.

1. **인코더의 출력과 디코더의 입력이 같은 크기**여야 한다(잠재 벡터를 주고받으므로).
2. **디코더의 출력이 입력과 같은 형태**여야 한다(입력을 정답으로 쓰므로).

그 사이는 자유롭다. 본문 p19가 "오토인코더는 모델을 구성하는 일부 계층이 아니라 **인코더와 디코더로
구성된 구조의 모델을 같은 입력과 정답으로 학습한 결과물을 가리키는 표현**"이라고 한 이유가 여기에 있다.
오토인코더는 특정 구조가 아니라 **학습 방식**의 이름이다.

| 구조 | 파라미터 수 | 최적 에포크 | 재현 오차 | 노이즈 0.2 오차 |
|---|---|---|---|---|
| A 대칭(본문) | 203,800 | 70 | 0.01855 | 0.04786 |
| B 인코더 깊음 | 320,216 | 62 | 0.01808 | 0.04357 |
| C 디코더 깊음 | 320,216 | 78 | **0.01679** | 0.04738 |
| **D 합성곱 인코더** | **119,640** | 80 | 0.01855 | **0.03965** |

**어느 쪽을 깊게 하는 것이 나은가**

B와 C는 파라미터 수가 320,216으로 똑같은데 **C(디코더를 깊게)가 조금 더 낫다**(0.01679 대 0.01808).
다만 차이가 크지 않고, 대칭 구조(A)와 견주어도 0.01855에서 0.01679로 9% 개선에 그친다.
**파라미터를 57% 더 쓴 대가치고는 작은 수익**이다.

그래서 **어디에 투자할지는 성능보다 목적으로 정하는 편이 낫다.**

- **인코더를 깊게** — 잠재 벡터의 품질이 중요할 때. 7-4절처럼 **인코더만 떼어 전이 학습에 쓸 계획**이라면
  인코더에 투자하는 것이 맞다. 디코더는 학습이 끝나면 버려진다.
- **디코더를 깊게** — 생성 품질이 중요할 때. 11장에서 다룰 생성 모델은 **잠재 공간에서 새 데이터를 만들어 내므로**
  디코더가 주인공이 된다.

본문 p20이 "특히 **인코더 객체만 떼어내 사용하는 경우가 많은데**"라고 한 것과 이어지는 이야기다.

**합성곱 인코더(D)가 가장 흥미롭다**

D는 **파라미터가 119,640개로 A보다 41% 적은데 재현 오차는 0.01855로 똑같다.**
그리고 **노이즈가 섞인 입력에서는 네 구조 중 가장 좋다**(0.03965).

이유는 5장에서 배운 그대로다. 다층 퍼셉트론 인코더는 784개 픽셀을 **1차원으로 늘어놓은 뒤** 전부 연결하므로
**2차원 구조 정보를 잃는다.** 본문 p23도 "2차원 이미지를 평탄화한 후 다층 퍼셉트론으로 학습하는 과정에서
이미지의 2차원 구조 정보가 훼손된 탓도 있다"고 지적했다. 합성곱 인코더는 이 손실이 없다.

**노이즈에 강한 것도 합성곱의 성질에서 온다.** 합성곱 필터는 이웃한 픽셀을 함께 보므로,
한 픽셀만 튀는 잡음은 주변에 묻혀 약해진다. **필터가 일종의 평활화 역할을 하는 셈**이다.

**구현에서 주의할 점이 하나 있다.** 데이터셋이 `(784,)` 형태를 내보내므로,
합성곱 계층에 넣으려면 **`(1, 28, 28)`로 되돌려야 한다.** 위 코드에서는 인코더 맨 앞에 `nn.Unflatten`을 넣어
**모델 안에서 처리**했다. 이렇게 하면 데이터 준비 과정을 건드리지 않아도 되고,
다른 모델들과 같은 데이터로더를 그대로 쓸 수 있다.

그리고 지문이 알려 준 대로 **마지막 합성곱 계층의 특징 지도를 평탄화**해야 한다.
`(B, 32, 7, 7)`을 `nn.Flatten()`으로 `(B, 1568)`로 만든 뒤 선형 계층으로 잠재 벡터 크기까지 줄인다.
합성곱만으로는 1차원 잠재 벡터를 만들 수 없으므로 **선형 계층이 반드시 하나 필요하다.**

**남은 비대칭 — 디코더는 여전히 다층 퍼셉트론이다.** D 모델은 인코더만 합성곱이라
구조가 극단적으로 비대칭인데도 네 구조 중 가장 효율이 좋다. 이 문제가 말하려는 바를 가장 잘 보여 주는 사례다.
디코더까지 합성곱으로 만드는 방법이 바로 다음 문제(7-10)의 주제다.

### 문제 검토

- **적절성: 적합. 문제의식이 정확하다.** "`MNISTAutoEncoder` 모델 때문에 … 대칭을 이뤄야 한다고
  **오해할 수도 있다**"는 도입이 좋다. 본문 예제가 우연히 대칭이었을 뿐인데 그것을 규칙으로 받아들이는
  흔한 오해를 정확히 겨눈다.
- **★★ [검토] 앞 두 항목의 '2개로 수정'이 무슨 뜻인지 알기 어렵다.**
  본문 [코드 7-5]의 인코더와 디코더는 **이미 각각 선형 계층이 2개**다(`784→128→z`, `z→128→784`).
  그런데 지문은 "인코더에 포함된 선형 계층을 **2개로 수정**", "디코더에 포함된 선형 계층을 **2개로 수정**"이라고 한다.
  **이미 2개인 것을 2개로 바꾸라는 말이 되어 독자가 멈춘다.**

  문맥상 의도는 **'한쪽만 계층 수를 바꿔 비대칭으로 만들어 보라'**로 읽힌다.
  가능성은 둘이다. (a) 원래 인코더를 3계층으로 상정한 초고에서 '2개로 줄이라'는 뜻이었거나,
  (b) '2개'가 '**3개**' 또는 '**하나 더**'의 오기이거나.
  → **어느 쪽이든 현재 문장으로는 실행할 수 없으므로 반드시 다시 써야 한다.**
- **[검토] 세 번째 항목의 안내가 정확하다.** "마지막 합성곱 계층이 출력하는 특징 지도를 평탄화해서
  1차원의 잠재 벡터로 변환해야 한다"가 정확한 힌트다. 합성곱만으로는 1차원 잠재 벡터를 만들 수 없다는
  핵심을 짚어 준다. 다만 **입력이 `(784,)` 형태라 `(1, 28, 28)`로 되돌려야 한다는 점**은 언급이 없어,
  형태 불일치 오류로 한 번 막힐 수 있다.
- **[검토] "필요하다면 잠재 벡터의 크기를 수정해도 좋다"가 친절하다.** 본문의 2는 비교하기에 너무 작아
  구조 차이가 잘 드러나지 않는다. 이 한마디가 그 제약을 풀어 준다.

**윤문안**

> **7-9** `MNISTAutoEncoder` 모델 때문에 오토인코더 속의 인코더와 디코더의 구조가 대칭을 이뤄야 한다고
> 오해할 수도 있다. [코드 7-5]의 인코더와 디코더는 각각 두 개의 선형 계층으로 이루어져 대칭을 이룬다.
> 다음과 같이 **한쪽 구조만 바꿔 비대칭으로 만든** 세 개의 오토인코더 모델을 만들어 보고 각각 결과를 확인해 보자.
> 필요하다면 잠재 벡터의 크기를 수정해도 좋다.
> - **인코더에 포함된 선형 계층만 세 개로 늘려 수정**
> - **디코더에 포함된 선형 계층만 세 개로 늘려 수정**
> - 인코더를 합성곱 신경망의 구조로 변경: 이 경우 **평탄화된 입력 텐서를 다시 `(1, 28, 28)` 형태로 되돌려야 하며**,
>   마지막 합성곱 계층이 출력하는 특징 지도를 평탄화해서 1차원의 잠재 벡터로 변환해야 한다.

## 연습 문제 7-10 [도전 문제]

> [연습 문제 7-9]에서 세 번째 항목으로 제안한 합성곱 계층으로 구성된 인코더처럼 디코더도 합성곱 계층을 사용해
> 만들 수 있다. 그런데 `nn.Conv2d`로 만든 합성곱 계층은 특징 지도의 크기를 늘릴 수 없어 합성곱 디코더를 만들 수 없다.
> 파이토치를 사용해 합성곱 디코더를 만들 때 특징 지도의 크기를 늘리는 데 사용되는 대표적인 두 가지 방법이 있다.
> 하나는 계층을 지날수록 정보를 확장하며 이미지를 복원하는 전치 합성곱 계층(`nn.ConvTranspose2d` 클래스로 생성)을
> 사용하는 방법이며, 다른 하나는 보간법으로 이미지의 크기를 늘리는 업샘플링 계층(`nn.Upsample` …)이다.

※ 본문 지문은 **[연습 문제 7-10]**이라고 되어 있으나 자기 자신을 가리키므로 **[연습 문제 7-9]**가 맞다(2단계 보고서 2번 항목).

In [11]:
class ConvTransposeAE(nn.Module):
    """전치 합성곱(nn.ConvTranspose2d)으로 크기를 늘리는 합성곱 오토인코더."""

    def __init__(self, z_size=Z_SIZE):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Unflatten(1, (1, 28, 28)),
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),    # -> (B, 16, 14, 14)
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),   # -> (B, 32,  7,  7)
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, z_size),
        )
        self.decoder = nn.Sequential(
            nn.Linear(z_size, 32 * 7 * 7), nn.ReLU(),
            nn.Unflatten(1, (32, 7, 7)),                            # -> (B, 32, 7, 7)
            # output_padding: 스트라이드 2로 크기를 두 배로 만들 때 한 칸이 모자라는 것을 채운다
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),  # -> (B, 1, 28, 28)
            nn.Sigmoid(),
            nn.Flatten(),                                           # -> (B, 784)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

class UpsampleAE(nn.Module):
    """업샘플링(nn.Upsample) + 합성곱으로 크기를 늘리는 합성곱 오토인코더."""

    def __init__(self, z_size=Z_SIZE):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Unflatten(1, (1, 28, 28)),
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, z_size),
        )
        self.decoder = nn.Sequential(
            nn.Linear(z_size, 32 * 7 * 7), nn.ReLU(),
            nn.Unflatten(1, (32, 7, 7)),
            nn.Upsample(scale_factor=2, mode='nearest'),            # -> (B, 32, 14, 14)
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),            # -> (B, 16, 28, 28)
            nn.Conv2d(16, 1, 3, padding=1),
            nn.Sigmoid(),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# 형태 확인
for name, model_class in [('전치 합성곱', ConvTransposeAE), ('업샘플링', UpsampleAE)]:
    model = model_class()
    with torch.no_grad():
        out = model(torch.zeros(2, 784))
    print(f'{name:>10}: 출력 {tuple(out.shape)}, 파라미터 {sum(p.numel() for p in model.parameters()):,}')

    전치 합성곱: 출력 (2, 784), 파라미터 36,233
      업샘플링: 출력 (2, 784), 파라미터 36,233


In [12]:
conv_decoders = {}
for name, model_class in [('전치 합성곱', ConvTransposeAE), ('업샘플링', UpsampleAE)]:
    torch.manual_seed(SEED)
    model = model_class()
    print(f'{name} 디코더 학습')
    result = train_autoencoder(model, label=f'[{name}] ')
    conv_decoders[name] = (model, result)

전치 합성곱 디코더 학습


  [전치 합성곱] 에포크   1 | 훈련 손실 0.06479 | 검증 손실 0.03595


  [전치 합성곱] 에포크  20 | 훈련 손실 0.01930 | 검증 손실 0.01960


  [전치 합성곱] 에포크  40 | 훈련 손실 0.01801 | 검증 손실 0.01841


  [전치 합성곱] 에포크  60 | 훈련 손실 0.01748 | 검증 손실 0.01796


...

  [업샘플링] 에포크   1 | 훈련 손실 0.05906 | 검증 손실 0.03250


  [업샘플링] 에포크  20 | 훈련 손실 0.01792 | 검증 손실 0.01814


  [업샘플링] 에포크  40 | 훈련 손실 0.01691 | 검증 손실 0.01740


  [업샘플링] 에포크  60 | 훈련 손실 0.01649 | 검증 손실 0.01701


  [업샘플링] 에포크  80 | 훈련 손실 0.01621 | 검증 손실 0.01678


In [13]:
print(f'{"디코더 구조":>16} {"파라미터 수":>12} {"최적 에포크":>11} {"재현 오차":>11} {"노이즈 0.2 오차":>16}')
print('-' * 72)
rows = [('다층 퍼셉트론(7-9 D)', structures['D 합성곱 인코더'][0], structures['D 합성곱 인코더'][1])]
rows += [(name, model, result) for name, (model, result) in conv_decoders.items()]
for name, model, result in rows:
    clean = reconstruction_error(model, test_loader)
    noisy = reconstruction_error(model, test_loader, 0.2)
    print(f'{name:>16} {sum(p.numel() for p in model.parameters()):12,d} '
          f'{result["best_epoch"]:11d} {clean:11.5f} {noisy:15.5f}')

          디코더 구조       파라미터 수      최적 에포크       재현 오차       노이즈 0.2 오차
------------------------------------------------------------------------


  다층 퍼셉트론(7-9 D)      119,640          80     0.01855         0.03965


          전치 합성곱       36,233          64     0.01735         0.04070


            업샘플링       36,233          79     0.01633         0.03570


### 풀이 해설

**왜 `nn.Conv2d`로는 크기를 늘릴 수 없는가**

합성곱 계층의 출력 크기 공식은 다음과 같다.

```
출력 = (입력 + 2 × 패딩 − 커널) // 스트라이드 + 1
```

스트라이드가 1 이상이므로 출력은 **입력보다 커질 수 없다**(패딩을 크게 주면 약간 커질 수는 있지만
두 배로 늘리려면 비현실적인 패딩이 필요하다). 그래서 별도의 방법이 필요하다.

**방법 1: 전치 합성곱(`nn.ConvTranspose2d`)**

합성곱을 **거꾸로 되짚는** 계층이다. 합성곱이 '여러 픽셀을 모아 하나로'라면 전치 합성곱은
'하나를 펼쳐 여러 픽셀로'다. 입력의 각 값에 커널을 곱해 출력 영역에 흩뿌린 뒤 겹치는 부분을 더한다.

**핵심은 커널 값이 학습 파라미터라는 점이다.** 즉 **어떻게 늘릴지를 모델이 배운다.**

구현에서 헷갈리는 것이 `output_padding`이다. `stride=2`로 크기를 두 배로 만들려 할 때
위 공식의 나눗셈이 버림이라 **7 → 13이 되어 한 칸이 모자란다.** `output_padding=1`이 이를 채워 14로 만든다.
**이 인자를 빼면 최종 출력이 28×28이 되지 않아 손실 계산에서 형태 오류가 난다.**

**방법 2: 업샘플링(`nn.Upsample`) + 합성곱**

먼저 **보간법으로 단순히 크기만 늘린 뒤**(`nearest`는 가장 가까운 값을 복사, `bilinear`는 선형 보간),
일반 합성곱으로 다듬는다. 확대 단계에는 **학습 파라미터가 없고**, 그다음 합성곱만 학습한다.

**두 방법의 비교**

| | 전치 합성곱 | 업샘플링 + 합성곱 |
|---|---|---|
| 확대 방식 | **학습한다** | 고정된 보간 규칙 |
| 계층 수 | 하나로 확대와 변환을 동시에 | 두 계층이 필요 |
| 알려진 단점 | **체커보드 무늬**가 생기기 쉽다 | 확대 방식이 고정되어 유연성이 떨어진다 |
| 구현 주의점 | `output_padding` 계산 | 없음(단순하다) |

**체커보드 무늬**는 전치 합성곱의 잘 알려진 부작용이다. 커널을 흩뿌릴 때 **겹치는 횟수가 위치마다 달라**
격자무늬가 생긴다. 이 때문에 실무에서는 업샘플링 + 합성곱을 선호하는 경우도 많다.

| 디코더 구조 | 파라미터 수 | 최적 에포크 | 재현 오차 | 노이즈 0.2 오차 |
|---|---|---|---|---|
| 다층 퍼셉트론(7-9 D) | 119,640 | 80 | 0.01855 | 0.03965 |
| 전치 합성곱 | **36,233** | 64 | 0.01735 | 0.04070 |
| 업샘플링 + 합성곱 | **36,233** | 79 | **0.01633** | **0.03570** |

**두 방법의 파라미터 수가 36,233개로 정확히 같다.** 우연이 아니다.
전치 합성곱은 `32→16`과 `16→1`의 3×3 커널을 학습하고, 업샘플링 방식도 확대 단계에 파라미터가 없는 대신
같은 크기의 합성곱 커널을 학습하기 때문이다.

**재현 오차는 업샘플링 쪽이 조금 낫고, 노이즈가 섞인 입력에서는 차이가 조금 더 벌어진다.**
다만 MNIST는 단순한 데이터라 이 정도 차이로 우열을 단정할 수는 없다. **둘 다 정답이며, 상황에 맞게 고르면 된다.**

**더 눈여겨볼 것은 다층 퍼셉트론 디코더와의 비교다.** 합성곱 디코더는 파라미터가 **3분의 1도 안 되는데**
(36,233 대 119,640) 재현 오차가 더 낮다. 인코더에 이어 디코더에서도 **2차원 구조를 살리는 것이 이득**이다.

**덧붙임 — 디코더 끝의 `nn.Flatten()`**

합성곱 디코더는 `(B, 1, 28, 28)`을 내보내는데 데이터셋이 `(B, 784)`를 정답으로 주므로,
손실을 계산하려면 형태를 맞춰야 한다. 위 코드에서는 디코더 마지막에 `nn.Flatten()`을 넣어 해결했다.
**반대로 데이터셋이 `(1, 28, 28)`을 내보내게 하고 인코더의 `nn.Unflatten`을 빼는 방법도 있다.**
합성곱을 쓰기로 했다면 후자가 더 자연스럽다.

**이 문제가 열어 주는 것**: 합성곱 디코더는 **11장의 생성 모델로 가는 다리**다.
잠재 벡터에서 이미지를 만들어 내는 구조가 바로 이것이며, DCGAN의 생성기가 전치 합성곱으로 만들어진다.

### 문제 검토

- **적절성: 도전 문제로 매우 적합하다.** 7-9의 세 번째 항목(합성곱 인코더)에서 자연스럽게 이어지고,
  **'왜 안 되는가'를 먼저 설명한 뒤 '그래서 이런 방법이 있다'로 넘어가는 구성**이 좋다.
  `nn.Conv2d`로는 크기를 늘릴 수 없다는 제약을 명확히 짚어 준 덕분에 독자가 헛수고하지 않는다.
- **★★ [검토] 참조 번호가 자기 자신을 가리킨다.** "**[연습 문제 7-10]**에서 세 번째 항목으로 제안한"은
  **[연습 문제 7-9]**여야 한다. 2단계 보고서 2번 항목 참조.
- **★ [검토] 무엇을 하라는 것인지가 문장으로 없다.** 지문은 두 방법을 **소개하는 것으로 끝난다.**
  "만들어 보자", "비교해 보자" 같은 **지시가 없어** 이것이 문제인지 설명인지 애매하다.
  7장의 다른 연습 문제는 모두 '~해 보자'로 끝나므로 이 문제만 형식이 다르다.
  → **마무리 문장을 반드시 넣어야 한다.**
- **[검토] 두 방법을 모두 알려 준 것은 도전 문제치고 친절한 편이다.** 다만 클래스 이름까지 주었으므로,
  **직접 구현할 때의 함정**(전치 합성곱의 `output_padding`)까지 미리 알려 줄 필요는 없다.
  `stride=2`로 크기를 두 배로 만들려 할 때 한 칸이 모자라는 것을 스스로 발견하는 것이
  도전 문제다운 경험이다. **현재 상태가 적절하다.**
- **[검토] 두 방법을 비교하게 하면 더 좋다.** 둘 다 만들어 보면 '어느 쪽이 나은가'가 궁금해지는데,
  실제로는 상황에 따라 다르다(전치 합성곱은 확대 방식을 학습하지만 체커보드 무늬가 생기기 쉽다).
  이 판단까지 가 보게 하면 도전 문제의 값어치가 커진다.

**윤문안**

> **7-10** [도전 문제] **[연습 문제 7-9]**에서 세 번째 항목으로 제안한 합성곱 계층으로 구성된 인코더처럼
> 디코더도 합성곱 계층을 사용해 만들 수 있다. 그런데 `nn.Conv2d`로 만든 합성곱 계층은 특징 지도의 크기를
> 늘릴 수 없어 합성곱 디코더를 만들 수 없다.
>
> 파이토치를 사용해 합성곱 디코더를 만들 때 특징 지도의 크기를 늘리는 데 사용되는 대표적인 두 가지 방법이 있다.
> 하나는 계층을 지날수록 정보를 확장하며 이미지를 복원하는 전치 합성곱 계층(`nn.ConvTranspose2d` 클래스로 생성)을
> 사용하는 방법이며, 다른 하나는 보간법으로 이미지의 크기를 늘리는 업샘플링 계층(`nn.Upsample` 클래스로 생성)을
> 사용하는 방법이다.
> **두 방법으로 각각 합성곱 디코더를 만들어 오토인코더를 완성하고, 재현 결과와 파라미터 수를 비교해 보자.**